In [75]:
import pandas as pd
import os
from tqdm import tqdm
from llm_asr_clarification import OpenAIWrapper
import json

DO_ALL_MEETINGS = True
# AMI_PATH = '/group/jrwhitehill/amicorpus'
AMI_PATH = '/home/pkongsomjit/Projects/llm_asr_clarification/datasets/amicorpus/xinlu_data'
MEETING_TO_DO= '/home/pkongsomjit/Projects/llm_asr_clarification/datasets/amicorpus/ES2005d'
# QUESTION_FILE = 'parsed_gt'
QUESTION_FILE = 'parsed_diarized_gt'
TRANSCRIPT_FILES = ['google_asr_diarized_transcript']#, 'qwen_transcript']
TRANSCRIPT_FILES = [f'score_using_{t}' for t in TRANSCRIPT_FILES]

# directories of meetings
if DO_ALL_MEETINGS:
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH) if entry.name not in ['ami_public_manual_1.6.2', 'xinlu_data']]
else:
    meeting_paths = [MEETING_TO_DO]

list_of_quiz_dicts = []
for meeting_path in tqdm(meeting_paths):
    question_path = os.path.join(meeting_path, "transcripts", f"quiz_from_{QUESTION_FILE}.json")
        
    # chatgpt = OpenAIWrapper()
    
    # Read question
    try:
        with open(question_path, "r", encoding="utf-8") as f:
            quiz = f.read()

        quiz = json.loads(quiz)
        meeting_name = meeting_path.split("/")[-1]
        for q in quiz:
            q['meeting_name'] = meeting_name
        list_of_quiz_dicts += quiz
    except:
        print(f"couldnt open file {question_path}")

df = pd.DataFrame(list_of_quiz_dicts)

100%|██████████| 16/16 [00:00<00:00, 17910.03it/s]


In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 5 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   question                                     160 non-null    str   
 1   correct_answer                               160 non-null    str   
 2   answer_using_google_asr_diarized_transcript  160 non-null    str   
 3   score_using_google_asr_diarized_transcript   160 non-null    object
 4   meeting_name                                 160 non-null    str   
dtypes: object(1), str(4)
memory usage: 6.4+ KB


In [77]:
for transcript_file in TRANSCRIPT_FILES:
    print(transcript_file)
    df = df[~df[transcript_file].str.contains('n/a', case=False, na=False)]
    df[transcript_file] = df[transcript_file].astype(int)
    df.info()

score_using_google_asr_diarized_transcript
<class 'pandas.DataFrame'>
Index: 157 entries, 0 to 159
Data columns (total 5 columns):
 #   Column                                       Non-Null Count  Dtype
---  ------                                       --------------  -----
 0   question                                     157 non-null    str  
 1   correct_answer                               157 non-null    str  
 2   answer_using_google_asr_diarized_transcript  157 non-null    str  
 3   score_using_google_asr_diarized_transcript   157 non-null    int64
 4   meeting_name                                 157 non-null    str  
dtypes: int64(1), str(4)
memory usage: 7.4 KB


In [79]:
for transcript_file in TRANSCRIPT_FILES:
    print(f"Global stats for {transcript_file}")
    df[transcript_file].describe()

Global stats for score_using_google_asr_diarized_transcript


In [81]:
stats_dict = {}
for transcript_file in TRANSCRIPT_FILES:
    print(f"Grouped stats for {transcript_file}")
    stats = df.groupby("meeting_name")[transcript_file].describe()
    print(stats)
    stats_dict[transcript_file] = stats

Grouped stats for score_using_google_asr_diarized_transcript
              count      mean       std  min   25%  50%  75%  max
meeting_name                                                     
EN2002a         9.0  0.666667  0.500000  0.0  0.00  1.0  1.0  1.0
EN2002b        10.0  0.500000  0.527046  0.0  0.00  0.5  1.0  1.0
EN2002c        10.0  0.700000  0.483046  0.0  0.25  1.0  1.0  1.0
EN2002d        10.0  0.900000  0.316228  0.0  1.00  1.0  1.0  1.0
ES2004a        10.0  0.700000  0.483046  0.0  0.25  1.0  1.0  1.0
ES2004b        10.0  0.800000  0.421637  0.0  1.00  1.0  1.0  1.0
ES2004c        10.0  0.800000  0.421637  0.0  1.00  1.0  1.0  1.0
ES2004d        10.0  0.800000  0.421637  0.0  1.00  1.0  1.0  1.0
IS1009a         9.0  0.555556  0.527046  0.0  0.00  1.0  1.0  1.0
IS1009b        10.0  0.600000  0.516398  0.0  0.00  1.0  1.0  1.0
IS1009c        10.0  1.000000  0.000000  1.0  1.00  1.0  1.0  1.0
IS1009d        10.0  0.900000  0.316228  0.0  1.00  1.0  1.0  1.0
TS3003a        

In [83]:
for transcript_file in TRANSCRIPT_FILES:
    print(f"Global stats by group for {transcript_file}")
    print(f"Mean across groups: {stats_dict[transcript_file]['mean'].mean()}")
    print(f"STD across groups: {stats_dict[transcript_file]['mean'].std()}")
    print(f"Min across groups: {stats_dict[transcript_file]['mean'].min()}")
    print(f"Max across groups: {stats_dict[transcript_file]['mean'].max()}")



Global stats by group for score_using_google_asr_diarized_transcript
Mean across groups: 0.76875
STD across groups: 0.13961845421433963
Min across groups: 0.5
Max across groups: 1.0
